<a href="https://colab.research.google.com/github/vikassingh0593/bytemaster_stocks/blob/dev/web_scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq > /dev/null  # Update package lists
!apt-get install openjdk-11-jdk-headless -qq > /dev/null # Install a supported JDK (OpenJDK 11 is recommended)

!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz #keep this
!tar xf spark-3.1.1-bin-hadoop3.2.tgz #keep this
!pip install -q findspark #keep this

import os
# Set JAVA_HOME to the correct path for OpenJDK 11
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"  # Correct path for OpenJDK 11
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2" #keep this

import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

# Test the Spark Session:
print(spark.version) # Print the Spark version to confirm it's working

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
3.1.1


In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pyspark.sql.functions import col, when, last, monotonically_increasing_id, lag, lead, coalesce, lit
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql import DataFrame

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lower
from pyspark.sql import functions as F

In [ ]:
# Install PySpark and yfinance
!pip install pyspark yfinance

import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

import os
from google.colab import drive
!pip install yfinance
# !pip install --upgrade numpy
from yfinance import Ticker
spark.conf.set("spark.sql.debug.maxToStringFields", 1000) # Or a higher value as needed

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [6]:
url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd=500033&qtrid=121.00&QtrName=March%202024"

# Define headers to mimic a real browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# Define cookies if required
cookies = {
    "cookie_name": "cookie_value"
}

# Send a GET request to the URL with headers and cookies
response = requests.get(url, headers=headers, cookies=cookies)

soup = BeautifulSoup(response.content, "html.parser")

rows = soup.find_all('tr')

In [23]:
# str(rows).split("</tr>")

In [50]:
print(list(rows)[16])
print(type(list(rows)[16]))
print(len(list(rows)[16]))

<tr></tr>
<class 'bs4.element.Tag'>
0


In [55]:
text_list = [td.get_text().strip() for td in rows[18].find_all('td')[1:8]]
print(text_list)

['B2) Institutions (Domestic)', '0', '0', '', '', '', '0.00']


In [61]:
k=0
i=0
data_lst = []
while k==0:
  try:
    text_list = [td.get_text().strip() for td in rows[i+17].find_all('td')[1:8]]
    print("#####################################################################", text_list)
    if text_list[0]=='B1) Institutions':
      append_flag = True
    if append_flag:
      data_lst.append(text_list)
    i+=1
  except:
    append_flag = False
    k+=1

##################################################################### ['B1) Institutions', '0', '0', '', '', '', '0.00']
##################################################################### ['B2) Institutions (Domestic)', '0', '0', '', '', '', '0.00']
##################################################################### ['Mutual Funds/', '1', '67000', '', '', '67,000', '0.51']
##################################################################### ['Alternate Investment Funds', '4', '45313', '', '', '45,313', '0.34']
##################################################################### ['Banks', '5', '1950', '', '', '1,950', '0.01']
##################################################################### ['Sub Total B1', '10', '114263', '', '', '1,14,263', '0.87']
##################################################################### ['B3) Institutions (Foreign)', '0', '0', '', '', '', '0.00']
##################################################################### ['Foreign Portfolio Investor

In [58]:
data_lst

[]

In [15]:
rows[7].find_all('td')
for td in rows[7].find_all('td'):
  print(td.get_text().strip())

ABB India Limited


In [ ]:
def rd_fn(Number):

    url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd={Number}&qtrid=121.00&QtrName=March%202024"

    # Define headers to mimic a real browser request
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    # Define cookies if required
    cookies = {
        "cookie_name": "cookie_value"
    }

    # Send a GET request to the URL with headers and cookies
    response = requests.get(url, headers=headers, cookies=cookies)

    soup = BeautifulSoup(response.content, "html.parser")

    rows = soup.find_all('tr')

    dataframe = []
    try:
        data_lst_title = []
        for td in rows[7].find_all('td'):
            data_lst_title.append(td.get_text().strip())

        for i in range(50):
            data_lst = []
            try:
                for td in rows[i+17].find_all('td'):
                    data_lst.append(td.get_text().strip())
                data_lst[0] = data_lst_title[0]
                if len(data_lst)==22:
                    # print(len(data_lst))
                    dataframe.append(data_lst)
                data_lst = []
            except:
                pass
    except:
        pass

    schema = StructType([
    StructField("title", StringType(), True),
    StructField("Category & Name of the Shareholders", StringType(), True),
    StructField("No. of shareholder", StringType(), True),
    StructField("No. of fully paid up equity shares held", StringType(), True),
    StructField("Blank_2", StringType(), True),
    StructField("Blank_3", StringType(), True),
    StructField("Total no. shares held", StringType(), True),
    StructField("Shareholding % calculated as per SCRR, 1957 As a % of (A+B+C2)", StringType(), True),
    StructField("Blank_4", StringType(), True),
    StructField("Blank_5", StringType(), True),
    StructField("Blank_6", StringType(), True),
    StructField("Blank_7", StringType(), True),
    StructField("No. of Voting Rights", StringType(), True),
    StructField("Total as a % of Total Voting right", StringType(), True),
    StructField("Blank_8", StringType(), True),
    StructField("Blank_9", StringType(), True),
    StructField("No. of Locked in shares-No.(a)", StringType(), True),
    StructField("No. of Locked in shares-As a % of total Shares held(b)", StringType(), True),
    StructField("No. of equity shares held in dematerialized form(Not Applicable)", StringType(), True),
    StructField("Sub-categorization of shares (XV)-Shareholding (No. of shares) under-SubCategory_I", StringType(), True),
    StructField("Sub-categorization of shares (XV)-Shareholding (No. of shares) under-SubCategory_II", StringType(), True),
    StructField("Sub-categorization of shares (XV)-Shareholding (No. of shares) under-SubCategory_III", StringType(), True)
    ])

    data_df = spark.createDataFrame(dataframe, schema)

    pyspark_df = data_df.withColumn(
        "Institution_name",
        when(col("`Category & Name of the Shareholders`") == "B1) Institutions", "B1) Institutions")
        .when(col("`Category & Name of the Shareholders`") == "B2) Institutions (Domestic)", "B2) Institutions (Domestic)")
        .when(col("`Category & Name of the Shareholders`") == "B3) Institutions (Foreign)", "B3) Institutions (Foreign)")
        .when(col("`Category & Name of the Shareholders`") == "B4) Central  Government/  State  Government(s)/ President of India", "B4) Central  Government/  State  Government(s)/ President of India")
        .when(col("`Category & Name of the Shareholders`") == "B5) Non-Institutions", "B5) Non-Institutions")
        .when(col("`Category & Name of the Shareholders`") == "B=B1+B2+B3+B4", "B=B1+B2+B3+B4")
        .otherwise("--")).withColumn("Number", lit(Number))

    # Add a unique identifier column
    pyspark_df = pyspark_df.withColumn("id", monotonically_increasing_id())

    # Define a window specification based on the unique identifier
    window_spec = Window.orderBy("id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

    # Use the 'last' function to fill missing values based on the last non-null value
    pyspark_df = pyspark_df.withColumn(
        "Institution_name",
        last(when(col("Institution_name") != "--", col("Institution_name")), ignorenulls=True).over(window_spec)
    )

    # Drop the intermediate unique identifier column
    pyspark_df = pyspark_df.drop("id")

    return pyspark_df